# 00 · Explore the datasets

**Run this first.** It answers one question: *do the adapters actually understand
the folder layout of the Kaggle mirrors you attached?*

Community mirrors sometimes reorganise the upstream structure. When they do,
every later step fails with "0 pairs" or an empty manifest, and the fix belongs
here — in `configs/datasets/*.yaml`, not in Python.

### Before running

1. **Add data** (right panel) → attach the datasets you want:
   - `brendanalvey/visdrone-dronevehicle` — DroneVehicle, the core UAV source
   - `monishshrivastava1/llvip-dataset` — LLVIP, night-time, tightly aligned
   - `samdazel/teledyne-flir-adas-thermal-dataset-v2` — FLIR ADAS, ground level
   - `pandrii000/hituav-a-highaltitude-infrared-thermal-dataset` — HIT-UAV, evaluation only
2. **Settings** → Accelerator: **GPU**, Internet: **ON**

Attaching a subset is fine; missing datasets are skipped, not fatal.

In [ ]:
# --- Pull the project code from GitHub -------------------------------------
# Requires "Internet" to be ON in the notebook settings panel on the right.
REPO_URL = "https://github.com/Astrq23/rgb-2-thermal.git"
BRANCH   = "main"

import os, subprocess, sys
from pathlib import Path

REPO_DIR = Path("/kaggle/working/rgb-2-thermal")

if REPO_DIR.exists():
    # Re-running the notebook: fast-forward instead of re-cloning.
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--depth", "1", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "reset", "--hard", f"origin/{BRANCH}"], check=True)
else:
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True
    )

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR / "src"))

print("repo:", REPO_DIR)
print("commit:", subprocess.run(
    ["git", "-C", str(REPO_DIR), "rev-parse", "--short", "HEAD"],
    capture_output=True, text=True).stdout.strip())

In [ ]:
# Editable install so `import rgb2thermal` works everywhere, including inside
# DataLoader worker processes. --no-deps keeps Kaggle's preinstalled torch.
!pip install -e . --no-deps -q

import torch
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

## What is actually mounted?

The tree below is ground truth. Compare it against the `search_roots` in each
`configs/datasets/*.yaml`.

In [ ]:
!python scripts/inspect_datasets.py --depth 4

### Reading the probe output

| Status | Meaning | What to do |
|---|---|---|
| `OK — N pairs` | The adapter found the data | Nothing |
| `NOT MOUNTED` | No search root matched | Attach the dataset, or add its path to `search_roots` |
| `root exists but 0 pairs matched` | Folder layout differs from expectations | Compare the tree above with the YAML and fix the globs |

For the last case, the usual fixes are all config-only:

- the RGB/thermal folders use unexpected names → add them to `thermal_tokens` / `rgb_tokens`
- the two modalities do not share a filename → set `key_regex`
- there is genuinely no modality folder → set `force_modality` (as HIT-UAV does)

## Build the unified manifest

In [ ]:
!python scripts/build_manifest.py --out /kaggle/working/manifest.csv

Check the printed table before going further:

- **Counts** — DroneVehicle should land near 28k pairs, LLVIP near 15k. Anything
  in the low hundreds means the adapter is only seeing part of the tree.
- **Split leakage** — a warning naming a dataset means its scenes appear in more
  than one split. That is expected when its own official split is honoured; if
  you would rather have strictly disjoint scenes, rebuild with
  `--set data.respect_split_hint=false`.

In [ ]:
import pandas as pd

manifest = pd.read_csv("/kaggle/working/manifest.csv", keep_default_na=False)
display(pd.crosstab(manifest["dataset"], manifest["split"], margins=True))
display(pd.crosstab(manifest["viewpoint"], manifest["time_of_day"], margins=True))

## Alignment check — the step people skip and regret

Paired training assumes the RGB and thermal frames show the same scene at the
same instant. DroneVehicle and LLVIP are hardware-aligned. **FLIR is not**: its
two cameras have different fields of view, and `preprocess.rgb.fov_crop` in
`configs/datasets/flir_v2.yaml` is an estimate, not a calibration.

A misaligned pair does not raise an error. It teaches the generator to blur, and
the loss curve looks perfectly healthy the whole time.

In [ ]:
!python scripts/check_alignment.py --manifest /kaggle/working/manifest.csv --n 3

from pathlib import Path
from IPython.display import Image as ShowImage, display

for path in sorted(Path("/kaggle/working/outputs/alignment").glob("*.png")):
    print(path.name)
    display(ShowImage(filename=str(path), width=1100))

**Read the 4th panel** (thermal edges in red over RGB):

- Edges land on buildings, vehicles, road markings → aligned, keep the dataset.
- Edges float away from structures → misaligned. Try a different crop:

  ```
  !python scripts/check_alignment.py --dataset flir_v2 --fov-crop 0.55 --n 3
  ```

  Sweep a few values, then write the best one into
  `configs/datasets/flir_v2.yaml`. If nothing lines up, set `enabled: false` —
  less data beats data that teaches blurring.

Next: **01_kaggle_train.ipynb**.